# FuelIQ — Model Training & Evaluation

Train and compare models for fuel consumption prediction:
1. Linear Regression (baseline)
2. Random Forest
3. Gradient Boosting

Evaluate with R2, MAE, RMSE, MAPE and cross-validation.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

from ml.src.features import engineer_features, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, TARGET
from ml.src.preprocessing import build_preprocessor

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load and prepare data
df = pd.read_csv('../data/synthetic_trips.csv')
df = engineer_features(df)
print(f'Dataset: {df.shape}')

feature_cols = NUMERICAL_FEATURES + CATEGORICAL_FEATURES
X = df[feature_cols]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# Train models
preprocessor = build_preprocessor()

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42),
}

results = {}
predictions = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model),
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    predictions[name] = y_pred
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = np.mean(np.abs((y_test - y_pred) / y_test.clip(lower=0.01))) * 100
    cv = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2')
    
    results[name] = {'R2': r2, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'CV_R2': cv.mean()}
    print(f'{name}: R2={r2:.4f}, MAE={mae:.2f}L, MAPE={mape:.1f}%')

results_df = pd.DataFrame(results).T
results_df

In [ ]:
# Model comparison bar charts
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
metrics = ['R2', 'MAE', 'RMSE', 'MAPE']
colors = ['#2ecc71', '#e74c3c', '#3498db']

for i, metric in enumerate(metrics):
    values = [results[m][metric] for m in models.keys()]
    bars = axes[i].bar(models.keys(), values, color=colors)
    axes[i].set_title(metric)
    axes[i].tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                     f'{val:.3f}' if metric != 'MAPE' else f'{val:.1f}%',
                     ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs Actual scatter plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, y_pred) in enumerate(predictions.items()):
    axes[i].scatter(y_test, y_pred, alpha=0.3, s=10)
    max_val = max(y_test.max(), y_pred.max())
    axes[i].plot([0, max_val], [0, max_val], 'r--', linewidth=2)
    axes[i].set_xlabel('Actual Fuel (L)')
    axes[i].set_ylabel('Predicted Fuel (L)')
    axes[i].set_title(f'{name}\nR2={results[name]["R2"]:.4f}')
    axes[i].set_xlim(0, max_val * 1.05)
    axes[i].set_ylim(0, max_val * 1.05)

plt.suptitle('Predicted vs Actual', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Residual plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, y_pred) in enumerate(predictions.items()):
    residuals = y_test.values - y_pred
    axes[i].scatter(y_pred, residuals, alpha=0.3, s=10)
    axes[i].axhline(y=0, color='r', linestyle='--')
    axes[i].set_xlabel('Predicted Fuel (L)')
    axes[i].set_ylabel('Residual (L)')
    axes[i].set_title(f'{name}')

plt.suptitle('Residual Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (Gradient Boosting)
best_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)),
])
best_pipeline.fit(X_train, y_train)

# Get feature names after preprocessing
num_features = NUMERICAL_FEATURES
cat_features = best_pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(CATEGORICAL_FEATURES)
all_features = list(num_features) + list(cat_features)

importances = best_pipeline.named_steps['model'].feature_importances_
feat_imp = pd.Series(importances, index=all_features).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
feat_imp.plot(kind='barh', color=sns.color_palette('viridis', len(feat_imp)))
plt.xlabel('Feature Importance')
plt.title('Gradient Boosting: Feature Importance')
plt.tight_layout()
plt.show()

print('Top 5 features:')
for feat, imp in feat_imp.tail(5).items():
    print(f'  {feat}: {imp:.4f}')

In [ ]:
# Save the best model
model_path = '../models/fuel_consumption_model.joblib'
joblib.dump(best_pipeline, model_path)
print(f'Model saved to {model_path}')

# Quick test prediction
test_input = pd.DataFrame([{
    'engine_size_l': 2.4,
    'cylinders': 4,
    'distance_km': 50.0,
    'duration_minutes': 60.0,
    'avg_speed_kmh': 50.0,
    'max_speed_kmh': 80.0,
    'idle_time_minutes': 5.0,
    'load_weight_kg': 500.0,
    'speed_variance': 1.6,
    'idle_ratio': 0.083,
    'distance_per_hour': 50.0,
    'fuel_type': 'diesel',
    'route_type': 'mixed',
}])

prediction = best_pipeline.predict(test_input)
print(f'\nTest prediction: {prediction[0]:.2f} L for 50km trip')
print(f'Efficiency: {50.0/prediction[0]:.2f} km/L')